[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Quantum/Quantum_for_Signal_Processors.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Quantum Computing for Signal Processors

A hype-resistant introduction with a home-field advantage: qubits are [unit vectors](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb), gates are unitary matrices, and the QFT — the algorithm behind quantum's most famous speedups — is *literally the FFT's matrix* ([verified](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) against `np.fft`). Everything simulated exactly in NumPy.

## 1. Pre-requisites

[Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) (unitary matrices, tensor structure helps), [Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) S7 (the DFT).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

# a state of n qubits = a unit vector in C^(2^n); gates = unitaries; measurement = |amplitude|²
def kron_all(mats):
    out = np.array([[1.0+0j]])
    for m in mats: out = np.kron(out, m)
    return out
I2 = np.eye(2); H = np.array([[1, 1], [1, -1]])/np.sqrt(2)
X = np.array([[0, 1], [1, 0]]);
def phase(theta): return np.diag([1, np.exp(1j*theta)])

---
### 🕐 Session 1 of 3 — *Qubits Are Vectors, Gates Are Unitaries* (~35 min)
**Goal:** the whole formalism in linear-algebra terms; entanglement as non-factorizability.
**Builds on:** [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb). &nbsp; **Feeds into:** Session 2 (the QFT).

---

<details><summary>🎓 <b>Teacher notes — Session 1: Qubits Are Vectors, Gates Are Unitaries</b></summary>

**The one idea to land before anything else:** an $n$-qubit state is *just* a unit vector in $\mathbb{C}^{2^n}$. That's it — no mysticism. A classical register of $n$ bits needs $n$ numbers to describe; a quantum register needs $2^n$ complex amplitudes. That exponential blow-up in *description size* is the entire hardware story — it's also exactly why classical simulation of quantum systems is hard, and why quantum computers, if you can build them, might exploit that same blow-up.

**Walk the vocabulary through students' existing linear-algebra vocabulary, not new jargon:**
- "Gate" = unitary matrix. Reversible, norm-preserving. Same object as a rotation matrix — think Parseval, not magic.
- "Measurement" = sampling index $k$ with probability $|\psi_k|^2$. This is the *only* nonlinear operation in the whole theory — everything else (gates) is linear algebra they already know.
- "Entanglement" = a state that is not a tensor product of smaller states. No new physics vocabulary needed: it's just "this vector isn't a Kronecker product of two shorter vectors."

**The demo cell is the whole lecture in one block.** `H` on qubit 0 creates a superposition; `CNOT` correlates qubit 1 with it. The state $\frac{1}{\sqrt2}(|00\rangle+|11\rangle)$ can't be written as $|\psi_1\rangle \otimes |\psi_2\rangle$ — and the SVD test makes that concrete rather than mystical: reshape the 4 amplitudes into a $2\times2$ matrix, and a product state is exactly a rank-1 matrix. Two nonzero singular values = provably not a product state. This SVD trick is worth dwelling on — it turns "spooky entanglement" into "the same rank test you'd run on any matrix."

**Common misconception to head off:** entanglement is *correlation*, not *communication*. Nothing here sends information faster than light — measuring qubit 0 tells you the outcome of qubit 1 with certainty, but only because the joint distribution was correlated from the start, exactly like a coin split into two envelopes (except the quantum version can't be explained by a hidden classical variable — that's Bell's theorem, out of scope here but worth a one-sentence mention if asked).

**Pacing:** the linear-algebra reframing is the whole point of Session 1 — don't let the class rush to Session 2's QFT before this sits. If Session 1 feels too short, have students hand-verify $H \otimes I_2$ acting on $|00\rangle$ by hand before running the cell.
</details>

## 2. No Mysticism Required

💡 **Intuition.** One qubit: a unit vector in $\mathbb{C}^2$. $n$ qubits: a unit vector in $\mathbb{C}^{2^n}$ — the exponential size of that space is the entire hardware story. Gates are [unitary matrices](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) (reversible, norm-preserving — Parseval's cousins); measurement samples index $k$ with probability $|\psi_k|^2$ and is the *only* nonlinear thing in the theory. **Entanglement** is simply a joint state that doesn't factor as a tensor product — correlation with no classical joint distribution behind it.

In [2]:
# build the Bell state with H then CNOT; verify it cannot factor
CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)
psi = np.zeros(4, complex); psi[0] = 1                      # |00⟩
psi = CNOT @ kron_all([H, I2]) @ psi
print("Bell state amplitudes:", psi.round(3), " → P(00)=P(11)=1/2, P(01)=P(10)=0")
# factorization test: a product state has rank-1 'amplitude matrix'
Mamp = psi.reshape(2, 2)
print(f"singular values of the amplitude matrix: {np.linalg.svd(Mamp, compute_uv=False).round(3)}")
print("→ TWO nonzero singular values: not rank-1 ⇒ genuinely entangled (the SVD detects it!)")

Bell state amplitudes: [0.707+0.j 0.   +0.j 0.   +0.j 0.707+0.j]  → P(00)=P(11)=1/2, P(01)=P(10)=0
singular values of the amplitude matrix: [0.707 0.707]
→ TWO nonzero singular values: not rank-1 ⇒ genuinely entangled (the SVD detects it!)


**What just happened.** The circuit produced amplitudes $(0.707, 0, 0, 0.707)$ — exactly $\frac{1}{\sqrt2}(|00\rangle+|11\rangle)$, so measurement gives 00 or 11 with 50/50 probability and 01/10 never happen. That's already suspicious-looking correlation, but the SVD is the proof: reshaping the length-4 amplitude vector into a $2\times2$ matrix and taking singular values gives **two** nonzero values ($0.707, 0.707$), not one. A product state $|\psi_1\rangle\otimes|\psi_2\rangle$ always reshapes to a rank-1 (one nonzero singular value) matrix — that's what a Kronecker product *is*, algebraically. Two nonzero singular values is a clean, checkable certificate that this state cannot be factored into two independent qubits: the correlation is baked into the amplitudes themselves, not layered on top of two separate stories.

---
### 🕐 Session 2 of 3 — *The QFT Is the FFT* (~40 min)
**Goal:** build the quantum Fourier transform from gates; verify it equals the DFT matrix exactly.
**Builds on:** Session 1; [Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) S7. &nbsp; **Feeds into:** Session 3 (what quantum actually speeds up).

---

<details><summary>🎓 <b>Teacher notes — Session 2: The QFT Is the FFT</b></summary>

**This is the payoff session for a DSP audience — land it hard.** The quantum Fourier transform is not "Fourier, but quantum-flavored" — it is *literally* the $2^n\times2^n$ DFT matrix, applied to a length-$2^n$ amplitude vector, built from $O(n^2)$ gates instead of $O(n)$ arithmetic operations. The `ORACLE` line in the code (`‖QFT circuit − DFT matrix‖∞ ≈ 3.8e-15`) is the class oracle: that residual is machine epsilon, not "close" — it's an equality proof, not an approximation.

**Make the FFT connection concrete, not just asserted:** the circuit's structure — Hadamards then controlled-phase gates then a bit-reversal permutation at the end — is line-for-line the radix-2 Cooley–Tukey butterfly diagram students may already know from Foundations 1 S7. If they've drawn a butterfly diagram before, draw the gate cascade next to it: same shape. The bit-reversal step at the end of `qft_circuit` is *the same* bit-reversal every FFT implementation needs — it isn't a quantum-specific quirk.

**The catch every pop-science article skips, and the one thing to insist students repeat back:** you cannot read out the transformed amplitude vector. Measurement gives *one* sample drawn with probability $|\hat\psi_k|^2$ — you get one number, not a spectrum, and you cannot re-run the same state to get more (measurement destroys it; you'd need to re-prepare from scratch). So a "quantum FFT" is *not* a fast way to compute an FFT of classical data you already have — that would need $O(2^n)$ just to load the data in the first place, erasing any speedup. The **only** place QFT helps is when a *global property* of the spectrum (like a period) can be extracted from a handful of measurements without ever reading out the full vector — which is exactly what the period-finding demo below shows, and exactly what Shor's algorithm exploits.

**If a student asks "so what's it good for":** defer to Session 3's scoreboard — this session's job is just the mechanism (QFT = DFT matrix, built from gates), not the payoff.
</details>

## 3. Home Turf

💡 **Intuition.** The QFT on $n$ qubits applies the $2^n \times 2^n$ **DFT matrix** to the amplitude vector — built from $O(n^2)$ two-qubit gates, the same divide-and-conquer as the [radix-2 FFT](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) (the gate cascade IS the butterfly diagram). The catch every headline omits: the result lives in *amplitudes you cannot read out directly* — measuring gives one sample, not the spectrum. QFT speedups exist only where a global *property* of the spectrum (like a period) suffices — which is exactly what Shor's algorithm extracts.

In [3]:
def qft_circuit(n_q):
    """QFT as a product of 1- and 2-qubit gates (returns the full unitary)."""
    N = 2**n_q
    U = np.eye(N, dtype=complex)
    for j in range(n_q):
        # H on qubit j
        U = kron_all([I2]*j + [H] + [I2]*(n_q-j-1)) @ U
        # controlled phases from qubits j+1..n
        for k in range(j+1, n_q):
            CP = np.eye(N, dtype=complex)
            for idx in range(N):
                bits = [(idx >> (n_q-1-b)) & 1 for b in range(n_q)]
                if bits[j] and bits[k]:
                    CP[idx, idx] = np.exp(2j*np.pi / 2**(k-j+1))
            U = CP @ U
    # bit-reversal permutation (same one as the FFT!)
    perm = [int(format(i, f"0{n_q}b")[::-1], 2) for i in range(N)]
    return U[perm]

n_q = 4
U_qft = qft_circuit(n_q)
F = np.array([[np.exp(2j*np.pi*i*k/2**n_q) for k in range(2**n_q)] for i in range(2**n_q)])/np.sqrt(2**n_q)
print(f"ORACLE: ‖QFT circuit − DFT matrix‖∞ = {np.abs(U_qft - F).max():.2e}")
assert np.abs(U_qft - F).max() < 1e-10
print(f"and against np.fft: ‖U_qft @ e₃ − ifft-convention column‖ = "
      f"{np.abs(U_qft[:, 3] - np.fft.ifft(np.eye(16)[3])*4).max():.2e}")

ORACLE: ‖QFT circuit − DFT matrix‖∞ = 3.78e-15
and against np.fft: ‖U_qft @ e₃ − ifft-convention column‖ = 1.49e-16


**What just happened.** Both checks bottom out at machine precision: `3.78e-15` against the hand-built DFT matrix, `1.49e-16` against `np.fft`. Those aren't "close" — for double-precision arithmetic accumulating $O(n^2)$ complex multiplications, error at the $10^{-15}$ level *is* zero; it's the same size as the rounding noise you'd get squaring and re-summing anything this many times. The gate cascade (Hadamards, controlled-phase rotations, then bit-reversal) computes the exact $2^n\times2^n$ DFT matrix — the same matrix Foundations 1 builds from butterflies, the same permutation every radix-2 FFT applies at the end. Nothing here is "quantum-flavored Fourier"; it's the DFT, factored into $O(n^2)$ two-qubit gates instead of computed by $O(n\log n)$ classical butterfly operations — and, per the discussion above, hidden inside a state you can only sample once.

In [4]:
# period finding — the heart of Shor — on a simulated register
n_q = 6; N = 2**n_q
r_period = 8
psi = np.zeros(N, complex)
psi[::r_period] = 1; psi /= np.linalg.norm(psi)             # a periodic state (post-oracle)
out = qft_circuit(n_q) @ psi
probs = np.abs(out)**2
plt.figure(figsize=(7.5, 2.4))
plt.stem(probs, basefmt=" ", markerfmt=".")
plt.title(f"measure after QFT: peaks at multiples of N/r = {N//r_period} → the period, from ONE global property")
plt.xlabel("measured value"); plt.tight_layout(); plt.show()
peaks = np.where(probs > 0.01)[0]
print(f"measurement outcomes: {[int(p) for p in peaks]} — spacing {N//r_period} reveals r = {r_period}")

measurement outcomes: [0, 8, 16, 24, 32, 40, 48, 56] — spacing 8 reveals r = 8


/tmp/ipykernel_319673/2780989073.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("measured value"); plt.tight_layout(); plt.show()


**What just happened.** The input state has amplitude only every $r=8$ positions out of $N=64$ — a period-8 "comb" in the register basis. After the QFT, measurement lands *only* at multiples of $N/r = 8$ (the printed outcomes: $0,8,16,\dots,56$), and every other position has exactly zero probability. This is the DSP fact in a quantum costume: the Fourier transform of a period-$r$ comb is itself a period-$(N/r)$ comb — sampling a periodic signal at its period concentrates all spectral energy at harmonics of the fundamental. **The catch this idealized demo hides:** it prepared the periodic state directly and used $r=8$, a power of two that divides $N=64$ exactly, so every peak lands exactly on an integer and the period is readable by eye. Real period-finding (as in Shor's algorithm) first has to *build* this periodic state from a modular-exponentiation oracle, and $r$ generally does *not* divide $N$ evenly — the peaks smear across neighboring bins, and recovering $r$ requires a continued-fractions post-processing step on the noisy measured value. This cell shows the clean endpoint of the idea, not the full algorithm.

---
### 🕐 Session 3 of 3 — *What Quantum Actually Speeds Up* (~30 min)
**Goal:** the honest scoreboard: where proofs exist, where hype lives, and what to watch.
**Builds on:** Session 2.

---

<details><summary>🎓 <b>Teacher notes — Session 3: What Quantum Actually Speeds Up</b></summary>

**This session's job is calibration, not a new demo.** Students walk in having just seen a genuinely exact QFT=DFT equivalence and a clean period-finding trick — the risk is they leave thinking "quantum computers are fast Fourier machines." Session 3 exists to correct that before it calcifies.

**Walk the scoreboard row by row and press on "proven" vs. "hoped for":**
- Shor's algorithm (factoring) has an exponential speedup *with a rigorous proof* — but needs millions of physical qubits with today's error-correction overhead, not the toy sizes in this notebook.
- Grover's search is *quadratic*, not exponential — a real but modest win, often erased in practice by the cost of building the oracle.
- Quantum simulation of quantum systems (chemistry, materials) is the least-hyped, most-defensible use case: simulating $n$ interacting quantum particles classically costs $O(2^n)$ *in principle*, and a quantum computer sidesteps that by being made of the same kind of stuff it's simulating.
- Generic ML/optimization: **no proven quantum advantage.** This is the row to spend the most time on, since it's the row most oversold in industry marketing. If data has to be loaded from classical memory, that loading step alone is often $\Omega(N)$ and erases any downstream speedup — the same "you can't read out the state for free" lesson from Session 2's QFT caveat, generalized.

**The number worth writing on the board:** ~1000 physical qubits per logical (error-corrected) qubit with current hardware. When a headline says "N-qubit quantum computer," ask whether N counts physical or logical qubits — that ratio is the whole gap between demos like this notebook and a cryptography-breaking machine.

**Close on the transfer, not the hype:** everything in this notebook — unitaries, interference, the DFT-as-QFT — is vocabulary students already own from linear algebra and DSP. The honest takeaway is "I can read a quantum-computing headline critically now," not "quantum computers are about to change everything."
</details>

## 4. The Scoreboard

| Problem | Speedup | Status |
|---|---|---|
| Factoring / discrete log (Shor) | exponential | proven; needs ~millions of good qubits |
| Unstructured search (Grover) | quadratic only | proven; modest in practice |
| Simulating quantum systems | exponential | the original killer app — chemistry/materials |
| Generic ML / optimization | — | **no proven advantage**; data loading often eats the win |

💡 **Intuition.** The honest summary for an engineer: quantum computers are *interference machines* — they win when a problem's answer can be encoded so wrong paths cancel ([the QFT's](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) specialty) — and today's hardware fights decoherence with error-correction overheads of ~1000 physical per logical qubit. Track logical-qubit counts, not press releases. Your DSP training transfers verbatim: unitaries, interference, transforms — you already speak the language.

---
## Where next

- [Foundations 1](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) S7 — the butterfly you just rebuilt from gates.
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) — the entire formalism, secretly.